# 04 · FAISS index comparison

The default pipeline uses an exact flat `IndexFlatIP` (cosine on L2-normalised
embeddings) — exact and fast at this scale. For large galleries you would move to
approximate indices. Here we compare **flat** vs **IVF** vs **HNSW** on the same
gallery embeddings: build time, search latency, and **recall@10** against the exact
flat result.

In [ ]:
import os, sys
PROJ = os.path.dirname(os.getcwd()) if os.path.split(os.getcwd())[1] == 'notebooks' else os.getcwd()
if PROJ not in sys.path: sys.path.insert(0, PROJ)
nb_dir = os.path.join(PROJ, 'notebooks')
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import utils
print('project root:', PROJ)


In [ ]:
import time, numpy as np
import faiss
P = utils.load_pipeline('configs/default.yaml')
engine = P['engine']
# Use the cached full optical embeddings as our candidate pool.
emb_all, _ = engine.cache_full_embeddings('optical')
N = 4000
X = emb_all[:N].astype('float32')        # L2-normalised already
rng = np.random.RandomState(0)
q_idx = rng.choice(N, size=500, replace=False)
Q = X[q_idx]
print('gallery:', X.shape, ' queries:', Q.shape)

In [ ]:
# Exact baseline (ground truth).
t0 = time.perf_counter(); flat = faiss.IndexFlatIP(X.shape[1]); flat.add(X)
gt_scores, gt_ids = flat.search(Q, 10)
build_ms = (time.perf_counter()-t0)*1000
print(f'flat build {build_ms:.1f} ms');

In [ ]:
def bench(name, index, prebuilt=False, nprobe=None, gt_ref=None):
    if not prebuilt:
        t0 = time.perf_counter(); index.train(X); index.add(X); build = (time.perf_counter()-t0)*1000
    else:
        build = 0.0  # flat is the pre-built exact baseline
    if nprobe is not None: index.nprobe = nprobe
    t0 = time.perf_counter()
    _, ids = index.search(Q, 10)
    lat = (time.perf_counter()-t0)/Q.shape[0]*1000  # ms per query
    recall = float((ids == gt_ref).any(axis=1).mean())*100 if gt_ref is not None else 100.0
    print(f'{name:<22} build {build:8.1f} ms   {lat*1000:7.0f} µs/q   recall@10 {recall:5.1f}%')
    return {'name': name, 'build_ms': build, 'us_per_q': lat*1000, 'recall10': recall}

res = []
d = X.shape[1]
res.append(bench('flat (exact)', flat, prebuilt=True, gt_ref=gt_ids))
res.append(bench('IVF, nlist=16', faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 16, faiss.METRIC_INNER_PRODUCT), nprobe=4, gt_ref=gt_ids))
res.append(bench('IVF, nlist=64', faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 64, faiss.METRIC_INNER_PRODUCT), nprobe=8, gt_ref=gt_ids))
hnsw = faiss.IndexHNSWFlat(d, 32); hnsw.hnsw.efConstruction = 200
res.append(bench('HNSW, M=32', hnsw, gt_ref=gt_ids))

### Discussion
The plot below shows the trade-off: IVF and HNSW cut search latency by an order of
magnitude with only a small recall loss at these gallery sizes. Flat stays exact.

In [ ]:
import matplotlib.pyplot as plt
names = [r['name'] for r in res]
lat   = [r['us_per_q'] for r in res]
rec   = [r['recall10'] for r in res]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(names, lat, color='#4f9cf9'); ax[0].set_title('Search latency (µs / query)');
ax[0].tick_params(axis='x', rotation=20); ax[0].set_yscale('log')
ax[1].bar(names, rec, color='#3ecf8e'); ax[1].set_title('Recall@10 vs exact (%)')
ax[1].tick_params(axis='x', rotation=20); ax[1].set_ylim(0, 105)
for a in ax: a.grid(axis='y', alpha=.3)
plt.tight_layout();

### Scaling note
For 100k+ vectors, persist an IVF/HNSW gallery and let `RetrievalEngine` reload it
from `faiss/` — see [05_database_and_persistence.ipynb](05_database_and_persistence.ipynb).